# KineCopilot - Batch Pose Precomputation

This notebook processes the Penn Action Dataset to extract pose data and save it as JSON files compatible with the KineCopilot API.

**Benefits of precomputation:**
- Avoid reprocessing videos on every analysis
- Enable fast agent-only experimentation
- Works great for research datasets

**Output:** ZIP file containing pose JSON files in `PoseSequence` schema format

## 1. Setup - Install Dependencies

In [26]:
!pip install mediapipe opencv-python tqdm matplotlib

In [27]:
import os
import json
import cv2
import kagglehub
import mediapipe as mp
import numpy as np
from pathlib import Path
from tqdm import tqdm
import zipfile
import matplotlib.pyplot as plt

## 2. Download Penn Action Dataset

**Penn Action Dataset** contains 2326 video sequences of 15 different actions.

### To add the dataset to your Kaggle notebook:

1. Go to https://www.kaggle.com/datasets
2. Search for "Penn Action Dataset" or use this slug: `dreamquark/penn-action`
3. Click "Add Data" in your Kaggle notebook
4. The dataset will be mounted at `/kaggle/input/penn-action/`

### Dataset Structure:
```
/kaggle/input/penn-action/
├── frames/
│   ├── 0001/          # Frame images for video 0001
│   ├── 0002/
│   └── ...
├── labels/
│   ├── 0001.mat       # Action labels (MATLAB format)
│   └── ...
└── videos/            # Not always present - we'll use frames
```

**Note:** If the exact dataset structure differs, adjust the paths below accordingly.

In [28]:
path = kagglehub.dataset_download("kaushalbora18/penn-action-dataset")
DATASET_ROOT = Path(path) / 'Penn_Action'  # ← add Penn_Action here
FRAMES_DIR = DATASET_ROOT / 'frames'
OUTPUT_DIR = Path('/kaggle/working/kinecopilot_poses')
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Define Helper Functions

These functions mirror the backend logic for pose extraction and angle computation.

In [29]:
# Landmark name mapping (same as backend)
LANDMARK_NAMES = {
    0: "NOSE", 1: "LEFT_EYE_INNER", 2: "LEFT_EYE", 3: "LEFT_EYE_OUTER",
    4: "RIGHT_EYE_INNER", 5: "RIGHT_EYE", 6: "RIGHT_EYE_OUTER",
    7: "LEFT_EAR", 8: "RIGHT_EAR", 9: "MOUTH_LEFT", 10: "MOUTH_RIGHT",
    11: "LEFT_SHOULDER", 12: "RIGHT_SHOULDER", 13: "LEFT_ELBOW",
    14: "RIGHT_ELBOW", 15: "LEFT_WRIST", 16: "RIGHT_WRIST",
    17: "LEFT_PINKY", 18: "RIGHT_PINKY", 19: "LEFT_INDEX", 20: "RIGHT_INDEX",
    21: "LEFT_THUMB", 22: "RIGHT_THUMB", 23: "LEFT_HIP", 24: "RIGHT_HIP",
    25: "LEFT_KNEE", 26: "RIGHT_KNEE", 27: "LEFT_ANKLE", 28: "RIGHT_ANKLE",
    29: "LEFT_HEEL", 30: "RIGHT_HEEL", 31: "LEFT_FOOT_INDEX", 32: "RIGHT_FOOT_INDEX"
}

def compute_joint_angle(a, b, c):
    """Compute angle at point b formed by a-b-c."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    ba = a - b
    bc = c - b
    
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    angle = np.arccos(cos_angle)
    return np.degrees(angle)

def compute_joint_angles(keypoints_dict):
    """Compute all joint angles from keypoints dictionary."""
    angles = {}
    
    def safe_angle(p1, p2, p3):
        try:
            if all(k in keypoints_dict for k in [p1, p2, p3]):
                return compute_joint_angle(keypoints_dict[p1], keypoints_dict[p2], keypoints_dict[p3])
        except:
            pass
        return None
    
    angle_configs = [
        ('left_knee', 'LEFT_HIP', 'LEFT_KNEE', 'LEFT_ANKLE'),
        ('right_knee', 'RIGHT_HIP', 'RIGHT_KNEE', 'RIGHT_ANKLE'),
        ('left_hip', 'LEFT_SHOULDER', 'LEFT_HIP', 'LEFT_KNEE'),
        ('right_hip', 'RIGHT_SHOULDER', 'RIGHT_HIP', 'RIGHT_KNEE'),
        ('left_elbow', 'LEFT_SHOULDER', 'LEFT_ELBOW', 'LEFT_WRIST'),
        ('right_elbow', 'RIGHT_SHOULDER', 'RIGHT_ELBOW', 'RIGHT_WRIST'),
        ('left_shoulder', 'LEFT_ELBOW', 'LEFT_SHOULDER', 'LEFT_HIP'),
        ('right_shoulder', 'RIGHT_ELBOW', 'RIGHT_SHOULDER', 'RIGHT_HIP'),
    ]
    
    for name, p1, p2, p3 in angle_configs:
        angle = safe_angle(p1, p2, p3)
        if angle is not None:
            angles[name] = angle
    
    # Spine lean
    try:
        if all(k in keypoints_dict for k in ['LEFT_SHOULDER', 'RIGHT_SHOULDER', 'LEFT_HIP', 'RIGHT_HIP']):
            ls = np.array(keypoints_dict['LEFT_SHOULDER'])
            rs = np.array(keypoints_dict['RIGHT_SHOULDER'])
            lh = np.array(keypoints_dict['LEFT_HIP'])
            rh = np.array(keypoints_dict['RIGHT_HIP'])
            
            mid_shoulder = (ls + rs) / 2
            mid_hip = (lh + rh) / 2
            spine_vector = mid_shoulder - mid_hip
            vertical = np.array([0, -1])
            
            cos_angle = np.dot(spine_vector, vertical) / (np.linalg.norm(spine_vector) + 1e-8)
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angles['spine_lean'] = np.degrees(np.arccos(cos_angle))
    except:
        pass
    
    return angles

print("Helper functions defined ✓")

Helper functions defined ✓


## 4. Batch Processing Pipeline

Process all videos in the dataset and extract pose sequences.

**Note:** This uses MediaPipe Vision Tasks API (v0.10.x+) which replaces the older `mp.solutions.pose` API. The pose landmarker model will be automatically downloaded on first run.

In [ ]:
def process_video_sequence(video_dir, sample_rate=3):
    """Process a single video sequence (directory of frames) using MediaPipe Vision Tasks API."""
    
    # Get all frame images
    frame_files = sorted(video_dir.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(video_dir.glob('*.png'))
    
    if not frame_files:
        return None
    
    # Initialize MediaPipe Pose using Vision Tasks API (for v0.10.x)
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    # Download model if not exists
    import urllib.request
    model_path = '/kaggle/working/pose_landmarker.task'
    if not Path(model_path).exists():
        print("Downloading pose landmarker model...")
        urllib.request.urlretrieve(
            'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task',
            model_path
        )
    
    # Create PoseLandmarker
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    frames_data = []
    fps = 30.0  # Assume 30fps
    
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        for frame_idx, frame_path in enumerate(frame_files):
            if frame_idx % sample_rate != 0:
                continue
            
            # Read frame
            image = cv2.imread(str(frame_path))
            if image is None:
                continue
            
            # Convert to MediaPipe Image
            rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
            
            # Detect pose
            detection_result = landmarker.detect(mp_image)
            
            if detection_result.pose_landmarks:
                # Extract keypoints from first detected pose
                landmarks = detection_result.pose_landmarks[0]
                
                keypoints_list = []
                keypoints_dict = {}
                
                for idx, landmark in enumerate(landmarks):
                    name = LANDMARK_NAMES.get(idx, f"LANDMARK_{idx}")
                    keypoints_list.append({
                        'name': name,
                        'x': landmark.x,
                        'y': landmark.y,
                        'visibility': landmark.visibility
                    })
                    keypoints_dict[name] = (landmark.x, landmark.y)
                
                # Compute angles
                joint_angles = compute_joint_angles(keypoints_dict)
                
                # Create frame data
                frame_data = {
                    'frame_index': frame_idx,
                    'timestamp_ms': (frame_idx / fps) * 1000,
                    'keypoints': keypoints_list,
                    'joint_angles': joint_angles
                }
                frames_data.append(frame_data)
    
    if not frames_data:
        return None
    
    # Compute summary stats
    joint_data = {}
    for frame in frames_data:
        for joint, angle in frame['joint_angles'].items():
            if joint not in joint_data:
                joint_data[joint] = []
            joint_data[joint].append(angle)
    
    summary_stats = {}
    for joint, angles in joint_data.items():
        if angles:
            summary_stats[joint] = {
                'min': float(np.min(angles)),
                'max': float(np.max(angles)),
                'mean': float(np.mean(angles)),
                'std': float(np.std(angles))
            }
    
    # Build PoseSequence
    pose_sequence = {
        'sport': 'general',
        'fps': fps,
        'total_frames': len(frames_data),
        'frames': frames_data,
        'summary_stats': summary_stats
    }
    
    return pose_sequence

print("Processing function defined ✓")

## 5. Run Batch Processing with Progress Bar

In [31]:
# Limit processing for testing (remove limit for full dataset)
MAX_VIDEOS = 50  # Process first 50 videos (set to None for all)

if FRAMES_DIR.exists():
    video_dirs = sorted([d for d in FRAMES_DIR.iterdir() if d.is_dir()])
    
    if MAX_VIDEOS:
        video_dirs = video_dirs[:MAX_VIDEOS]
    
    print(f"Processing {len(video_dirs)} video sequences...")
    
    processed_count = 0
    failed_count = 0
    
    for video_dir in tqdm(video_dirs, desc="Processing videos"):
        video_id = video_dir.name
        output_path = OUTPUT_DIR / f"{video_id}_poses.json"
        
        # Skip if already processed
        if output_path.exists():
            processed_count += 1
            continue
        
        try:
            pose_sequence = process_video_sequence(video_dir, sample_rate=3)
            
            if pose_sequence:
                with open(output_path, 'w') as f:
                    json.dump(pose_sequence, f, indent=2)
                processed_count += 1
            else:
                failed_count += 1
        except Exception as e:
            print(f"\nError processing {video_id}: {e}")
            failed_count += 1
    
    print(f"\n✓ Processing complete!")
    print(f"  Successfully processed: {processed_count}")
    print(f"  Failed: {failed_count}")
else:
    print("⚠️ Dataset not found. Please add Penn Action Dataset to this notebook.")

Processing 50 video sequences...


Processing videos: 100%|██████████| 50/50 [00:00<00:00, 434.79it/s]


Error processing 0001: module 'mediapipe' has no attribute 'solutions'

Error processing 0002: module 'mediapipe' has no attribute 'solutions'

Error processing 0003: module 'mediapipe' has no attribute 'solutions'

Error processing 0004: module 'mediapipe' has no attribute 'solutions'

Error processing 0005: module 'mediapipe' has no attribute 'solutions'

Error processing 0006: module 'mediapipe' has no attribute 'solutions'

Error processing 0007: module 'mediapipe' has no attribute 'solutions'

Error processing 0008: module 'mediapipe' has no attribute 'solutions'

Error processing 0009: module 'mediapipe' has no attribute 'solutions'

Error processing 0010: module 'mediapipe' has no attribute 'solutions'

Error processing 0011: module 'mediapipe' has no attribute 'solutions'

Error processing 0012: module 'mediapipe' has no attribute 'solutions'

Error processing 0013: module 'mediapipe' has no attribute 'solutions'

Error processing 0014: module 'mediapipe' has no attribute 'sol

## 6. Export as ZIP

In [32]:
# Create ZIP file
zip_path = Path('/kaggle/working/kinecopilot_poses.zip')

json_files = list(OUTPUT_DIR.glob('*.json'))

if json_files:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for json_file in tqdm(json_files, desc="Creating ZIP"):
            zipf.write(json_file, arcname=json_file.name)
    
    print(f"\n✓ ZIP file created: {zip_path}")
    print(f"  Size: {zip_path.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"  Files: {len(json_files)}")
else:
    print("No JSON files found to zip.")

No JSON files found to zip.


## 7. Sample Visualization

Visualize pose overlay on a sample frame.

In [33]:
# Load a sample pose JSON
json_files = list(OUTPUT_DIR.glob('*.json'))

if json_files:
    sample_json = json_files[0]
    
    with open(sample_json, 'r') as f:
        sample_pose = json.load(f)
    
    print(f"Sample: {sample_json.name}")
    print(f"Sport: {sample_pose['sport']}")
    print(f"Total frames: {sample_pose['total_frames']}")
    print(f"FPS: {sample_pose['fps']}")
    print(f"\nJoint angles (first frame):")
    
    if sample_pose['frames']:
        first_frame = sample_pose['frames'][0]
        for joint, angle in first_frame['joint_angles'].items():
            print(f"  {joint}: {angle:.1f}°")
    
    # Visualize skeleton on first frame
    video_id = sample_json.stem.replace('_poses', '')
    video_dir = FRAMES_DIR / video_id
    
    if video_dir.exists():
        frame_files = sorted(video_dir.glob('*.jpg'))
        if not frame_files:
            frame_files = sorted(video_dir.glob('*.png'))
        
        if frame_files:
            image = cv2.imread(str(frame_files[0]))
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Draw keypoints
            if sample_pose['frames']:
                keypoints = sample_pose['frames'][0]['keypoints']
                h, w = image_rgb.shape[:2]
                
                for kp in keypoints:
                    x = int(kp['x'] * w)
                    y = int(kp['y'] * h)
                    cv2.circle(image_rgb, (x, y), 5, (0, 255, 0), -1)
            
            plt.figure(figsize=(12, 8))
            plt.imshow(image_rgb)
            plt.title(f"Sample Pose Visualization - {video_id}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()
else:
    print("No pose JSON files found for visualization.")

No pose JSON files found for visualization.


## 8. Download Instructions

The `kinecopilot_poses.zip` file is now ready in `/kaggle/working/`.

**To use with KineCopilot API:**

1. Download the ZIP file from Kaggle
2. Extract the JSON files locally
3. Use with the API:

```bash
curl -X POST http://localhost:8000/analyze \
  -F "pose_json=@0001_poses.json" \
  -F "sport=general" \
  -F "athlete_level=amateur"
```

This skips the video processing step and goes straight to coaching analysis!